In [1]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path
from typing import Any, Dict, Optional, Union

import numpy as np
import torch
from scipy.spatial.distance import pdist

from falkon import LogisticFalkon
from falkon.kernels import GaussianKernel
from falkon.gsc_losses import WeightedCrossEntropyLoss
from falkon.options import FalkonOptions

import matplotlib.pyplot as plt



ConfigLike = Dict[str, Any]
ConfigInput = Union[ConfigLike, str, Path]


class LogFalkonNPLM:
    """
    NPLM test statistic via LogisticFalkon classifier.

    Input API:
      - X: (N, d) pooled sample
      - y: (N,) or (N,1) labels in {0, 1}
           0 = reference (count N_R)
           1 = data      (count N_D)

    Notation:
      - N_R : number of reference points (y == 0)
      - N_D : number of data points      (y == 1)
      - NR  : expected data size under H0 (from config)
      - weight = NR / N_R

    Statistic:
      t = 2 * ( weight * sum_{y=0}(1 - exp(s_i)) + sum_{y=1}(s_i) )
    where s_i are LogisticFalkon outputs (scores/logits).
    """

    DEFAULT_CONFIG: ConfigLike = {
        # Kernel / Nyström
        "sigma": None,          # REQUIRED (float) for compute_statistic
        "M": "sqrt",            # int or "sqrt"

        # Regularization / solver
        "lambda": [1e-6],
        "iter": [1_000_000],
        "cg_tol": 1e-7,
        "keops": "no",

        # Optional consistency checks (if provided)
        "N_R": None,
        "N_D": None,

        # NPLM size under H0
        "NR": None,             # REQUIRED

        # Execution
        "seed": None,
        "cpu": False,
        "verbose": 1,           # 0=silent, 1=info
    }

    def __init__(self, config: Optional[ConfigInput] = None, output_path: Optional[Union[str, Path]] = None):
        self.output_path = str(output_path) if output_path is not None else None
        if self.output_path is not None:
            os.makedirs(self.output_path, exist_ok=True)

        cfg_dict = self._load_config(config) if config is not None else {}
        self.config = self._resolve_config(cfg_dict)

        self.model: Optional[LogisticFalkon] = None

        # resolved per run
        self.N_R: Optional[int] = None
        self.N_D: Optional[int] = None
        self.NR: Optional[float] = None
        self.weight: Optional[float] = None

        self._log("\n[NPLM] Initialized with config:")
        if self.config.get("verbose", 0) > 0:
            for k, v in self.config.items():
                print(f"  {k}: {v}")

    # ---------------- logging ----------------

    def _log(self, msg: str) -> None:
        if int(self.config.get("verbose", 0)) > 0:
            print(msg)

    # ---------------- config ----------------

    @staticmethod
    def _load_config(config: ConfigInput) -> ConfigLike:
        if isinstance(config, dict):
            return dict(config)

        path = Path(config)
        if not path.exists():
            raise FileNotFoundError(f"Config file not found: {path}")

        text = path.read_text(encoding="utf-8")
        suffix = path.suffix.lower()

        if suffix == ".json":
            return json.loads(text)
        if suffix in (".yml", ".yaml"):
            try:
                import yaml
            except ImportError as e:
                raise ImportError("YAML config requires `pip install pyyaml`.") from e
            return yaml.safe_load(text)

        raise ValueError("Config must be dict, .json, .yml or .yaml")

    @classmethod
    def _resolve_config(cls, user_cfg: ConfigLike) -> ConfigLike:
        cfg = dict(cls.DEFAULT_CONFIG)
        cfg.update(user_cfg)

        if isinstance(cfg["keops"], bool):
            cfg["keops"] = "yes" if cfg["keops"] else "no"

        if not isinstance(cfg["lambda"], (list, tuple)):
            cfg["lambda"] = [cfg["lambda"]]
        if not isinstance(cfg["iter"], (list, tuple)):
            cfg["iter"] = [cfg["iter"]]

        cfg["verbose"] = int(cfg.get("verbose", 0))

        # Require NR always
        if cfg.get("NR", None) is None:
            raise ValueError("Config must specify NR (expected data size under H0).")

        # sigma is required for compute_statistic; validated there (allows init without sigma if you only use estimator)
        if cfg.get("sigma", None) is not None:
            sigma = float(cfg["sigma"])
            if not np.isfinite(sigma) or sigma <= 0:
                raise ValueError(f"Config sigma must be positive finite; got sigma={sigma}")

        return cfg

    # ---------------- utility: sigma estimation ----------------

    @staticmethod
    def estimate_sigma_median(X: np.ndarray, max_points: int = 5000, seed: Optional[int] = None) -> float:
        """
        Estimate Gaussian bandwidth sigma using the median heuristic:
            sigma = median_{i<j} ||x_i - x_j||_2

        Intended use: call once on a representative sample, then store in config["sigma"].
        """
        X = np.asarray(X)
        if X.ndim != 2:
            raise ValueError("X must be 2D array (N, d)")
        X = np.ascontiguousarray(X)

        n = X.shape[0]
        if n < 2:
            raise ValueError("Need at least 2 points to estimate sigma")

        max_points = int(max_points)
        if max_points < 2:
            raise ValueError("max_points must be >= 2")

        if n > max_points:
            rng = np.random.default_rng(None if seed is None else int(seed))
            idx = rng.choice(n, size=max_points, replace=False)
            X = X[idx]

        d = pdist(X)  # Euclidean distances
        sigma = float(np.median(d))
        if not np.isfinite(sigma) or sigma <= 0:
            raise ValueError(f"Median heuristic returned invalid sigma={sigma}. Consider standardizing inputs.")
        return sigma

    # ---------------- helpers ----------------

    @staticmethod
    def _sqrt_rule(n: int) -> int:
        return int(np.sqrt(n))

    def _set_seed(self) -> None:
        seed = self.config.get("seed", None)
        if seed is None:
            self._log("[NPLM] No seed specified (non-reproducible run)")
            return
        seed = int(seed)
        self._log(f"[NPLM] Setting global seed = {seed}")
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    @staticmethod
    def _normalize_labels_01(y: np.ndarray) -> np.ndarray:
        """
        Ensure y is 1D array with values in {0,1}.
        """
        y = np.asarray(y)
        if y.ndim == 2 and y.shape[1] == 1:
            y = y.reshape(-1)
        if y.ndim != 1:
            raise ValueError("y must be shape (N,) or (N,1)")
        vals = np.unique(y)
        if not np.all(np.isin(vals, [0, 1])):
            raise ValueError(f"y must contain only 0 and 1; got unique values {vals}")
        # Keep float64 for Falkon cross-entropy
        return y.astype(np.float64, copy=False)

    def _resolve_sizes_from_labels(self, y01: np.ndarray) -> None:
        N_R = int(np.sum(y01 == 0))
        N_D = int(np.sum(y01 == 1))

        if N_R <= 0 or N_D <= 0:
            raise ValueError(f"Need both classes present: got N_R={N_R}, N_D={N_D}")

        if self.config["N_R"] is not None and int(self.config["N_R"]) != N_R:
            raise ValueError(f"Config N_R={self.config['N_R']} but labels imply N_R={N_R}")
        if self.config["N_D"] is not None and int(self.config["N_D"]) != N_D:
            raise ValueError(f"Config N_D={self.config['N_D']} but labels imply N_D={N_D}")

        self.N_R = N_R
        self.N_D = N_D
        self.NR = float(self.config["NR"])
        self.weight = self.NR / float(self.N_R)

        self._log(f"[NPLM] Label counts: N_R={self.N_R} (y=0), N_D={self.N_D} (y=1)")
        self._log(f"[NPLM] weight = NR/N_R = {self.NR}/{self.N_R} = {self.weight:.6g}")

    # ---------------- model ----------------

    def build_model(self) -> None:
        if self.N_R is None or self.N_D is None or self.weight is None:
            raise RuntimeError("Internal state not initialized. Call compute_statistic() first.")

        sigma = self.config.get("sigma", None)
        if sigma is None:
            raise ValueError(
                "Config must specify sigma (float) to train the model. "
                "You can estimate it once with `estimate_sigma_median` and then set config['sigma']."
            )
        sigma = float(sigma)

        kernel = GaussianKernel(sigma)

        M_cfg = self.config["M"]
        if isinstance(M_cfg, str):
            if M_cfg != "sqrt":
                raise ValueError("config['M'] must be an int or the string 'sqrt'")
            M = self._sqrt_rule(self.N_R + self.N_D)
        else:
            M = int(M_cfg)

        self._log(f"[NPLM] Using fixed sigma = {sigma:.6g}")
        self._log(f"[NPLM] Nyström centers M = {M}")

        opts = FalkonOptions(
            cg_tolerance=float(self.config["cg_tol"]),
            keops_active=str(self.config["keops"]),
            use_cpu=bool(self.config["cpu"]),
            debug=False,
        )

        self.model = LogisticFalkon(
            kernel=kernel,
            penalty_list=self.config["lambda"],
            iter_list=self.config["iter"],
            M=M,
            options=opts,
            loss=WeightedCrossEntropyLoss(kernel, neg_weight=float(self.weight)),
            seed=self.config.get("seed", None),
        )

    # ---------------- statistic ----------------

    @staticmethod
    def _compute_t(scores: torch.Tensor, y01: np.ndarray, weight: float) -> float:
        """
        scores: (N, 1) or (N,) torch tensor
        y01: (N,) numpy float/int with values {0,1}
        """
        if scores.ndim == 2 and scores.shape[1] == 1:
            scores = scores.reshape(-1)

        mask_ref = (y01 == 0)
        mask_data = (y01 == 1)

        mask_ref_t = torch.from_numpy(mask_ref)
        mask_data_t = torch.from_numpy(mask_data)

        s_ref = scores[mask_ref_t]
        s_data = scores[mask_data_t]

        diff = float(weight) * torch.sum(1.0 - torch.exp(s_ref))
        t = 2.0 * (diff + torch.sum(s_data))
        return float(t.item())

    def compute_statistic(self, X: np.ndarray, y: np.ndarray, return_details: bool = False):
        """
        Compute NPLM statistic from pooled X and 0/1 labels y.

        X: (N, d)
        y: (N,) or (N,1), values in {0,1}
        """
        X = np.asarray(X)
        if X.ndim != 2:
            raise ValueError("X must be 2D array (N, d)")
        X = np.ascontiguousarray(X)

        y01 = self._normalize_labels_01(y)
        if y01.shape[0] != X.shape[0]:
            raise ValueError("X and y must have the same number of rows")

        self._log("\n[NPLM] Starting computation of test statistic")
        self._set_seed()
        self._resolve_sizes_from_labels(y01)

        X_t = torch.from_numpy(X)  # zero-copy CPU view, torch.float32 if X is float32
        y_t = torch.from_numpy(y01.reshape(-1, 1)).to(dtype=X_t.dtype)

        self.build_model()

        self._log("[NPLM] Training LogisticFalkon...")
        t0 = time.time()
        self.model.fit(X_t, y_t)
        train_time = time.time() - t0
        self._log(f"[NPLM] Training finished in {train_time:.2f} s")

        scores = self.model.predict(X_t)
        t = self._compute_t(scores, y01, float(self.weight))

        self._log(f"[NPLM] Test statistic t = {t:.6g}")

        if not return_details:
            return t

        scores_flat = scores.reshape(-1)
        mask_ref_t = torch.from_numpy(y01 == 0)
        s_ref = scores_flat[mask_ref_t]
        Nw = float((float(self.weight) * torch.sum(torch.exp(s_ref))).item())

        return t, {
            "Nw": Nw,
            "train_time": float(train_time),
            "weight": float(self.weight),
            "sigma": float(self.config["sigma"]),
            "seed": self.config.get("seed", None),
            "scores": scores.detach().cpu().numpy(),
            "resolved_config": dict(self.config),
            "N_R": int(self.N_R),
            "N_D": int(self.N_D),
        }


In [7]:
from __future__ import annotations

import time
import warnings
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


@dataclass
class TuningResult:
    raw: pd.DataFrame
    summary: pd.DataFrame


class NPLMFalkonBootstrapTuner:
    """
    Bootstrap hyperparameter scan for NPLM + Falkon.

    For each (lambda, M) pair, repeated H0 tests are performed using only
    a large reference dataset.

    In each trial:
      - n_dat is fixed to n_bkg or drawn from Poisson(n_bkg)
      - ref and data are sampled jointly from the empirical reference
        distribution with replacement
      - the runner returns the scalar NPLM test statistic

    Outputs:
      - average test statistic vs M at varying lambda
      - average runtime vs M at varying lambda
    """

    def __init__(
        self,
        X_ref_large: np.ndarray,
        sigma: float,
        lambda_list: Sequence[float],
        M_list: Sequence[int],
        n_ref: int,
        n_bkg: int,
        n_trials: int,
        runner: Callable[[np.ndarray, np.ndarray, Dict[str, Any]], float],
        poisson_fluctuate_n_bkg: bool = True,
        random_state: Optional[int] = 0,
        runner_kwargs: Optional[Dict[str, Any]] = None,
        bootstrap_fraction_warning_threshold: float = 0.3,
    ) -> None:

        self.X_ref_large = np.asarray(X_ref_large)
        if self.X_ref_large.ndim == 1:
            self.X_ref_large = self.X_ref_large[:, None]

        self.sigma = float(sigma)
        self.lambda_list = list(lambda_list)
        self.M_list = list(M_list)

        self.n_ref = int(n_ref)
        self.n_bkg = int(n_bkg)
        self.n_trials = int(n_trials)

        self.runner = runner
        self.poisson_fluctuate_n_bkg = poisson_fluctuate_n_bkg
        self.runner_kwargs = {} if runner_kwargs is None else dict(runner_kwargs)

        self.bootstrap_fraction_warning_threshold = bootstrap_fraction_warning_threshold

        self.rng = np.random.default_rng(random_state)

        self._warned_totals: set[int] = set()

        self.raw_results_: Optional[pd.DataFrame] = None
        self.summary_: Optional[pd.DataFrame] = None

    def _sample_size_dat(self) -> int:
        if self.poisson_fluctuate_n_bkg:
            return max(1, int(self.rng.poisson(self.n_bkg)))
        return self.n_bkg

    def _check_joint_fraction(self, n_dat: int) -> None:
        n_tot = self.n_ref + n_dat
        n_pool = len(self.X_ref_large)

        frac = n_tot / n_pool

        if (
            frac > self.bootstrap_fraction_warning_threshold
            and n_tot not in self._warned_totals
        ):
            warnings.warn(
                f"n_ref + n_dat = {n_tot} is {frac:.1%} of the reference pool "
                f"(N_pool={n_pool}). Bootstrap samples may have limited diversity.",
                UserWarning,
            )
            self._warned_totals.add(n_tot)

    def _sample_joint(self, n_dat: int) -> Tuple[np.ndarray, np.ndarray]:
        n_tot = self.n_ref + n_dat

        idx = self.rng.choice(len(self.X_ref_large), size=n_tot, replace=True)

        idx_ref = idx[: self.n_ref]
        idx_dat = idx[self.n_ref :]

        return self.X_ref_large[idx_ref], self.X_ref_large[idx_dat]

    def _run_one_trial(self, lam: float, M: int, trial: int) -> Dict[str, Any]:

        n_dat = self._sample_size_dat()

        self._check_joint_fraction(n_dat)

        X_ref, X_dat = self._sample_joint(n_dat)

        cfg = {
            "sigma": self.sigma,
            "lambda": lam,
            "M": M,
            **self.runner_kwargs,
        }

        t0 = time.perf_counter()

        stat = self.runner(X_ref, X_dat, cfg)

        runtime = time.perf_counter() - t0

        return {
            "lambda": lam,
            "M": M,
            "trial": trial,
            "t_nplm": float(stat),
            "runtime_sec": runtime,
            "n_dat": n_dat,
        }

    def run(self) -> TuningResult:

        rows: List[Dict[str, Any]] = []

        for lam in self.lambda_list:
            for M in self.M_list:
                for trial in range(self.n_trials):

                    rows.append(self._run_one_trial(lam, M, trial))

        raw = pd.DataFrame(rows)

        summary = (
            raw.groupby(["lambda", "M"], as_index=False)
            .agg(
                n_trials=("trial", "count"),
                t_mean=("t_nplm", "mean"),
                t_std=("t_nplm", "std"),
                time_mean=("runtime_sec", "mean"),
                time_std=("runtime_sec", "std"),
            )
            .sort_values(["lambda", "M"])
        )

        summary["t_std"] = summary["t_std"].fillna(0)
        summary["time_std"] = summary["time_std"].fillna(0)

        summary["t_sem"] = summary["t_std"] / np.sqrt(summary["n_trials"])
        summary["time_sem"] = summary["time_std"] / np.sqrt(summary["n_trials"])

        self.raw_results_ = raw
        self.summary_ = summary

        return TuningResult(raw=raw, summary=summary)

    def plot_statistic_vs_M(self) -> Tuple[plt.Figure, plt.Axes]:

        if self.summary_ is None:
            raise RuntimeError("Call run() first.")

        fig, ax = plt.subplots(figsize=(7, 5))

        for lam, df in self.summary_.groupby("lambda"):

            df = df.sort_values("M")

            ax.plot(
                df["M"],
                df["t_mean"],
                marker="o",
                label=f"λ={lam:g}",
            )

        ax.set_xlabel("M")
        ax.set_ylabel("Average test statistic")
        ax.set_title("Average NPLM statistic vs M")

        ax.grid(True, alpha=0.3)
        ax.legend()

        return fig, ax

    def plot_runtime_vs_M(self) -> Tuple[plt.Figure, plt.Axes]:

        if self.summary_ is None:
            raise RuntimeError("Call run() first.")

        fig, ax = plt.subplots(figsize=(7, 5))

        for lam, df in self.summary_.groupby("lambda"):

            df = df.sort_values("M")

            ax.plot(
                df["M"],
                df["time_mean"],
                marker="o",
                label=f"λ={lam:g}",
            )

        ax.set_xlabel("M")
        ax.set_ylabel("Average runtime [s]")
        ax.set_title("Average runtime vs M")

        ax.grid(True, alpha=0.3)
        ax.legend()

        return fig, ax

In [8]:
from pathlib import Path
from typing import Any, Dict, Optional

import numpy as np


def run_single_nplm_test(
    X_ref: np.ndarray,
    X_dat: np.ndarray,
    cfg: Dict[str, Any],
    output_path: Optional[str | Path] = None,
) -> float:
    """
    Run one NPLM two-sample test using LogFalkonNPLM.

    Parameters
    ----------
    X_ref : np.ndarray
        Reference sample of shape (N_R, d).
    X_dat : np.ndarray
        Data/null sample of shape (N_D, d).
    cfg : dict
        Configuration dictionary. Must contain at least:
            - "sigma"
            - "lambda"
            - "M"
            - "NR"
        It may also contain optional Falkon/NPLM settings such as:
            - "iter"
            - "cg_tol"
            - "keops"
            - "cpu"
            - "seed"
            - "verbose"
    output_path : str or Path, optional
        Optional output path passed to LogFalkonNPLM.

    Returns
    -------
    float
        Scalar NPLM test statistic.
    """
    X_ref = np.asarray(X_ref)
    X_dat = np.asarray(X_dat)

    if X_ref.ndim == 1:
        X_ref = X_ref[:, None]
    if X_dat.ndim == 1:
        X_dat = X_dat[:, None]

    if X_ref.ndim != 2 or X_dat.ndim != 2:
        raise ValueError("X_ref and X_dat must be 2D arrays.")
    if X_ref.shape[1] != X_dat.shape[1]:
        raise ValueError(
            f"Feature mismatch: X_ref has d={X_ref.shape[1]}, "
            f"X_dat has d={X_dat.shape[1]}."
        )
    if len(X_ref) == 0 or len(X_dat) == 0:
        raise ValueError("X_ref and X_dat must both be non-empty.")

    # Build pooled sample
    X = np.ascontiguousarray(np.concatenate([X_ref, X_dat], axis=0), dtype=np.float32)

    # Build labels: 0 = reference, 1 = data
    y = np.concatenate([
        np.zeros(len(X_ref), dtype=np.float64),
        np.ones(len(X_dat), dtype=np.float64),
    ])

    # Build the config expected by LogFalkonNPLM
    nplm_config = {
        "sigma": float(cfg["sigma"]),
        "M": cfg["M"],
        "lambda": cfg["lambda"],
        "NR": float(cfg["NR"]),          # expected size under H0
        "N_R": int(len(X_ref)),          # optional consistency check
        "N_D": int(len(X_dat)),          # optional consistency check
        "iter": cfg.get("iter", [1_000_000]),
        "cg_tol": cfg.get("cg_tol", 1e-7),
        "keops": cfg.get("keops", "no"),
        "cpu": cfg.get("cpu", False),
        "seed": cfg.get("seed", None),
        "verbose": cfg.get("verbose", 0),
    }

    model = LogFalkonNPLM(config=nplm_config, output_path=output_path)

    t_nplm = model.compute_statistic(X=X, y=y, return_details=False)

    return float(t_nplm)

In [9]:
def sample_ref_exp(N, rate=8.0, xmax=1.0, rng=None, dtype=np.float32):
    """Sample 1D truncated exponential data with shape (N, 1)."""
    rng = np.random.default_rng() if rng is None else rng
    Z = 1.0 - np.exp(-rate * xmax)
    u = rng.random(int(N))
    x = -(1.0 / rate) * np.log(1.0 - u * Z)
    return x.astype(dtype, copy=False).reshape(-1, 1)


In [14]:
# Build a large 1D exponential reference pool
seed = 123
rng = np.random.default_rng(seed)

ref_large = sample_ref_exp(1000000, rate=8.0, xmax=1.0, rng=rng)
print("ref_large shape:", ref_large.shape)
print("first rows:")
print(ref_large[:5])


ref_large shape: (1000000, 1)
first rows:
[[0.14326133]
 [0.00691306]
 [0.0311035 ]
 [0.02546511]
 [0.02417487]]


In [11]:
# Choose sigma externally
sigma = LogFalkonNPLM.estimate_sigma_median(ref_large, max_points=5_000, seed=seed)
print(f"Estimated sigma (median heuristic): {sigma:.6f}")


Estimated sigma (median heuristic): 0.086531


In [17]:
# Hyperparameter scan setup
lambda_list = [1e-5, 1e-6]
M_list = [250, 500, 1000]

n_ref = 20000
n_bkg = 2000
n_trials = 3

tuner = NPLMFalkonBootstrapTuner(
    X_ref_large=ref_large,
    sigma=sigma,
    lambda_list=lambda_list,
    M_list=M_list,
    n_ref=n_ref,
    n_bkg=n_bkg,
    n_trials=n_trials,
    runner=run_single_nplm_test,
    poisson_fluctuate_n_bkg=True,
    random_state=seed,
    runner_kwargs={
        "NR": float(n_bkg),
        "iter": [20_000],
        "cg_tol": 1e-7,
        "keops": "no",
        "cpu": False,
        "verbose": 0,
        "seed": seed,
    },
)


In [18]:
result = tuner.run()
result.summary


Iteration 0 - penalty 1.000000e-05 - sub-iterations 20000


KeyboardInterrupt: 

In [ ]:
fig1, ax1 = tuner.plot_statistic_vs_M()
plt.show()

fig2, ax2 = tuner.plot_runtime_vs_M()
plt.show()
